# Dynamic Data Download - S5P, MODIS, CAMS

This notebook downloads satellite data with settings from **config.py**.

**To change settings:**
1. Edit `config.py` to change dates and paths
2. Run this notebook

**No need to edit this notebook!**

In [1]:
# Import necessary libraries
import osgeo
import ee
import xarray as xr
import geopandas as gpd
import geemap
import rioxarray as rxr
import odc.geo.xr
from datetime import datetime
import cdsapi
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Load configuration
%run 00_config.ipynb
print_config()

CURRENT CONFIGURATION

Date Range:
  Start: 2025-12-01
  End: 2025-12-31

Processing Settings:
  Sample Period: 1ME
  Statistical Method: mean
  Region: All
  Pollutants: SO2, NO2, CO, O3, PM2P5, PM10

Output Path:
  C:\projects\aqi_pipline_2026-01-08\03_Output_Data\monthly_data


In [3]:
# Authenticate and initialize Earth Engine
print("Initializing Earth Engine...")
ee.Authenticate()
ee.Initialize(project='scad-aqi-474811',
    opt_url='https://earthengine-highvolume.googleapis.com')
print("✓ Earth Engine initialized")

Initializing Earth Engine...
✓ Earth Engine initialized


In [9]:
# Load region boundaries
print("Loading region boundaries...")
gdf = gpd.read_file(PATHS['boundary_vector'])
fc = geemap.geopandas_to_ee(gdf)
print(f"✓ Loaded {len(gdf)} regions")

Loading region boundaries...
✓ Loaded 3 regions


In [10]:
# Define download path
download_path = PATHS['download']
download_path.mkdir(parents=True, exist_ok=True)

# Convert dates to datetime.date objects
from datetime import datetime
start_date = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d').date()
end_date = datetime.strptime(CONFIG['end_date'], '%Y-%m-%d').date()

print(f"\nDownload configuration:")
print(f"  Download path: {download_path}")
print(f"  Start date: {start_date}")
print(f"  End date: {end_date}")


Download configuration:
  Download path: C:\projects\aqi_pipline_2026-01-08\01_Downloaded
  Start date: 2025-12-01
  End date: 2025-12-31


## Define Download Functions

In [11]:
# S5P pollutant mapping
s5p_dict = {
    'SO2': ['COPERNICUS/S5P/OFFL/L3_SO2', 'SO2_column_number_density'],
    'NO2': ['COPERNICUS/S5P/OFFL/L3_NO2', 'NO2_column_number_density'],
    'CO': ['COPERNICUS/S5P/OFFL/L3_CO', 'CO_column_number_density'],
    'O3': ['COPERNICUS/S5P/OFFL/L3_O3', 'O3_column_number_density']
}

def download_s5p(pollutant, start_date, end_date, download_path, fc):
    """
    Downloads Sentinel-5P data for a given pollutant and time range.
    """
    print(f'\nDownloading Sentinel-5P {pollutant}...')
    print(f'  Date range: {start_date} to {end_date}')
    
    # Create ImageCollection
    ic = ee.ImageCollection(s5p_dict[pollutant][0]).filterDate(
        start_date.isoformat(), 
        end_date.isoformat()
    )
    
    # Open as xarray and download
    da = xr.open_dataset(
        ic,
        engine='ee',
        crs='EPSG:3857',
        scale=1100,
        geometry=fc.geometry(),
        fast_time_slicing=True
    )[s5p_dict[pollutant][1]].chunk('auto')
    
    # Save to Zarr
    output_file = download_path / f'S5P_{pollutant}_{start_date.isoformat()}_{end_date.isoformat()}.zarr'
    da.to_zarr(str(output_file))
    print(f'  ✓ Saved to: {output_file.name}')

def download_modis(start_date, end_date, download_path, fc):
    """
    Downloads MODIS MCD19A2 AOD data for a given time range.
    """
    print(f'\nDownloading MODIS AOD...')
    print(f'  Date range: {start_date} to {end_date}')
    
    # Create ImageCollection
    ic = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES').filterDate(
        start_date.isoformat(), 
        end_date.isoformat()
    )
    
    # Open as xarray and download
    da = xr.open_dataset(
        ic,
        engine='ee',
        crs='EPSG:3857',
        scale=1000,
        geometry=fc.geometry(),
        fast_time_slicing=True
    ).Optical_Depth_055.chunk('auto')
    
    # Save to Zarr
    output_file = download_path / f'MODIS_AOD055_{start_date.isoformat()}_{end_date.isoformat()}.zarr'
    da.to_zarr(str(output_file))
    print(f'  ✓ Saved to: {output_file.name}')

def download_cams(start_date, end_date, download_path):
    """
    Downloads CAMS global reanalysis data for a given time range.
    """
    print(f'\nDownloading CAMS data...')
    print(f'  Date range: {start_date} to {end_date}')
    
    # Create CDS API client
    client = cdsapi.Client()
    dataset = 'cams-global-reanalysis-eac4'
    
    # Iterate through years
    for year in range(start_date.year, end_date.year + 1):
        output_file = download_path / f'cams-global-reanalysis-eac4_{year}-{year+1}.grib'
        
        if output_file.exists():
            print(f'  ⊗ {year} already exists, skipping')
            continue
        
        # Define request
        request = {
            'variable': [
                'particulate_matter_2.5um',
                'particulate_matter_10um',
                'total_column_carbon_monoxide',
                'total_column_nitrogen_dioxide',
                'total_column_ozone',
                'total_column_sulphur_dioxide'
            ],
            'date': f'{start_date.isoformat()}/{end_date.isoformat()}',
            'time': ['00:00', '03:00', '06:00', '09:00', '12:00', '15:00', '18:00', '21:00'],
            'data_format': 'grib',
            'area': [51.529, 22.631, 56.382, 26.283]  # Abu Dhabi region
        }
        
        # Retrieve and download
        print(f'  Downloading year {year}...')
        ret = client.retrieve(dataset, request)
        ret.download(str(output_file))
        print(f'  ✓ Saved to: {output_file.name}')

print("Download functions defined.")

Download functions defined.


## Download Sentinel-5P Data

In [12]:
# Download S5P data for gas pollutants
print("=" * 70)
print("DOWNLOADING SENTINEL-5P DATA")
print("=" * 70)

s5p_pollutants = ['SO2', 'NO2', 'CO', 'O3']

for pollutant in s5p_pollutants:
    download_s5p(pollutant, start_date, end_date, download_path, fc)

print("\n✓ Sentinel-5P download complete!")

DOWNLOADING SENTINEL-5P DATA

  Date range: 2025-12-01 to 2025-12-31
  ✓ Saved to: S5P_SO2_2025-12-01_2025-12-31.zarr

  Date range: 2025-12-01 to 2025-12-31
  ✓ Saved to: S5P_NO2_2025-12-01_2025-12-31.zarr

  Date range: 2025-12-01 to 2025-12-31
  ✓ Saved to: S5P_CO_2025-12-01_2025-12-31.zarr

  Date range: 2025-12-01 to 2025-12-31
  ✓ Saved to: S5P_O3_2025-12-01_2025-12-31.zarr

✓ Sentinel-5P download complete!


## Download MODIS Data

In [13]:
# Download MODIS AOD data
print("\n" + "=" * 70)
print("DOWNLOADING MODIS AOD DATA")
print("=" * 70)

download_modis(start_date, end_date, download_path, fc)

print("\n✓ MODIS download complete!")


DOWNLOADING MODIS AOD DATA

  Date range: 2025-12-01 to 2025-12-31
  ✓ Saved to: MODIS_AOD055_2025-12-01_2025-12-31.zarr

✓ MODIS download complete!


## Download CAMS Data

In [15]:
# Download CAMS data
print("\n" + "=" * 70)
print("DOWNLOADING CAMS DATA")
print("=" * 70)

download_cams(start_date, end_date, download_path)

print("\n✓ CAMS download complete!")


DOWNLOADING CAMS DATA

  Date range: 2025-12-01 to 2025-12-31


HTTPError: 400 Client Error: Bad Request for url: https://ads.atmosphere.copernicus.eu/api/retrieve/v1/processes/cams-global-reanalysis-eac4/execution
invalid request
Request has not produced a valid combination of values, please check your selection.
{'variable': ['particulate_matter_2.5um', 'particulate_matter_10um', 'total_column_carbon_monoxide', 'total_column_nitrogen_dioxide', 'total_column_ozone', 'total_column_sulphur_dioxide'], 'date': '2025-12-01/2025-12-31', 'time': ['00:00', '03:00', '06:00', '09:00', '12:00', '15:00', '18:00', '21:00'], 'area': [51.529, 22.631, 56.382, 26.283], 'grid': '0.75/0.75'}

## Summary

In [ ]:
# Print download summary
print("\n" + "=" * 70)
print("DOWNLOAD SUMMARY")
print("=" * 70)
print(f"\nDate Range: {start_date} to {end_date}")
print(f"Download Location: {download_path}")

# List downloaded files
print("\nDownloaded Files:")
zarr_files = list(download_path.glob('*.zarr'))
grib_files = list(download_path.glob('*.grib'))

print(f"\nS5P & MODIS (Zarr):")
for f in sorted(zarr_files):
    print(f"  - {f.name}")

print(f"\nCAMS (GRIB):")
for f in sorted(grib_files):
    print(f"  - {f.name}")

print(f"\nTotal files: {len(zarr_files) + len(grib_files)}")
print("=" * 70)